In [ ]:
import yaml, json, os
import numpy as np
from datetime import datetime, timedelta

import src.parameters as parameters
import plotter.colors as colors
from src.models.requests import Trace
from src.models.orchestrator import Orchestrator

In [ ]:
import json, math
def load_traces(orchestrator, scenario, period, workload, schedulers, seeds, lc_weights):
    scenario_path_out = f"experiments/out/scenario_{scenario}/{period['start']}-{period['end']}/workload/{workload}/{period['step']}"
    
    time_range = orchestrator.simulation_time_range
    traces = {}
    carbon_footprints = {
        "actual": {},
        "forecast": {}
    }
    water_footprints = {
        "actual": {},
        "forecast": {}
    }
    land_use_footprints = {
        "actual": {},
        "forecast": {}
    }
    str_out = ""
    for scheduler in schedulers:
        for seed in seeds:
            #if scheduler == "RP": continue
            if scheduler == "G":
                lc_weights_str = [""]
            else:
                lc_weights_str = ["_"+str(w) for w in lc_weights]
            for weights in lc_weights_str:
                file_name = f"e_{seed}_{scheduler}{weights}.json"
                #file_name = f"e_{seed}_{scheduler}.json"
                with open(os.path.join(scenario_path_out, file_name), 'r') as f:
                    method = f"{scheduler}{weights}"
                    if method not in traces: traces[method] = {}
                    if method not in carbon_footprints["actual"]: carbon_footprints["actual"][method] = {}
                    if method not in carbon_footprints["forecast"]: carbon_footprints["forecast"][method] = {}
                    if method not in water_footprints["actual"]: water_footprints["actual"][method] = {}
                    if method not in water_footprints["forecast"]: water_footprints["forecast"][method] = {}
                    if method not in land_use_footprints["actual"]: land_use_footprints["actual"][method] = {}
                    if method not in land_use_footprints["forecast"]: land_use_footprints["forecast"][method] = {}

                    traces[method][seed] = json.load(f)["traces"]
                    carbon_footprints["actual"][method][seed] = [.0]*len(time_range.get_timestamps())
                    carbon_footprints["forecast"][method][seed] = [.0]*len(time_range.get_timestamps())
                    water_footprints["actual"][method][seed] = [.0]*len(time_range.get_timestamps())
                    water_footprints["forecast"][method][seed] = [.0]*len(time_range.get_timestamps())
                    land_use_footprints["actual"][method][seed] = [.0]*len(time_range.get_timestamps())
                    land_use_footprints["forecast"][method][seed] = [.0]*len(time_range.get_timestamps())
                    
                    for name, trace_compressed in traces[method][seed]:
                        trace = Trace(time_range)
                        trace.reload_compressed_trace(trace_compressed)
                        trace = trace.get_uncompressed_trace(orchestrator)
                        
                        for t, _ in enumerate(time_range.get_timestamps()):
                            carbon_footprints["actual"][method][seed][t] += trace["carbon_intensity_actual_raw"][t]*trace["execution_energy_kWh"][t]
                            carbon_footprints["forecast"][method][seed][t] += trace["carbon_intensity_forecast_raw"][t]*trace["execution_energy_kWh"][t]
                            water_footprints["actual"][method][seed][t] += trace["water_intensity_actual_raw"][t]*trace["execution_energy_kWh"][t]
                            water_footprints["forecast"][method][seed][t] += trace["water_intensity_forecast_raw"][t]*trace["execution_energy_kWh"][t]
                            land_use_footprints["actual"][method][seed][t] += trace["land_use_intensity_actual_raw"][t]*trace["execution_energy_kWh"][t]
                            land_use_footprints["forecast"][method][seed][t] += trace["land_use_intensity_forecast_raw"][t]*trace["execution_energy_kWh"][t]



    all_algorithms = list(carbon_footprints["actual"].keys())

    y_data_carbon_forecast = {
        algorithm: [] for algorithm in all_algorithms
    }
    y_data_water_forecast = {
        algorithm: [] for algorithm in all_algorithms
    }
    y_data_land_use_forecast = {
        algorithm: [] for algorithm in all_algorithms
    }
    y_data_carbon_actual = {
        algorithm: [] for algorithm in all_algorithms
    }
    y_data_water_actual = {
        algorithm: [] for algorithm in all_algorithms
    }
    y_data_land_use_actual = {
        algorithm: [] for algorithm in all_algorithms
    }
    for algorithm in all_algorithms:
        y_data_carbon_forecast[algorithm] = np.array(list(vals for vals in carbon_footprints["forecast"][algorithm].values())).T.tolist()
        y_data_water_forecast[algorithm] = np.array(list(vals for vals in water_footprints["forecast"][algorithm].values())).T.tolist()
        y_data_land_use_forecast[algorithm] = np.array(list(vals for vals in land_use_footprints["forecast"][algorithm].values())).T.tolist()
        y_data_carbon_actual[algorithm] = np.array(list(vals for vals in carbon_footprints["actual"][algorithm].values())).T.tolist()
        y_data_water_actual[algorithm] = np.array(list(vals for vals in water_footprints["actual"][algorithm].values())).T.tolist()
        y_data_land_use_actual[algorithm] = np.array(list(vals for vals in land_use_footprints["actual"][algorithm].values())).T.tolist()

    return {
            "carbon": y_data_carbon_actual,
            "water": y_data_water_actual,
            "land_use": y_data_land_use_actual,
        },{
            "carbon": y_data_carbon_forecast,
            "water": y_data_water_forecast,
            "land_use": y_data_land_use_forecast
        }

In [ ]:


scenario = 1
workload = "spark"
period =  {
    "start": "2024-01-15T00:00:00Z",
    "end": "2024-01-22T00:00:00Z",
    "step": 1800
}
schedulers = ["G", "R", "RP"]
seeds = [1,2]
weights = [
    "[1.0, 0.0, 0.0]",
    "[0.0, 1.0, 0.0]",
    "[0.0, 0.0, 1.0]",
    "[0.333, 0.333, 0.334]"

]
#config_file = f"experiments/scenarios/{scenario}.yaml"

#with open(config_file, "r") as f:
#    config = yaml.safe_load(f)


path_in_dc = f"experiments/in/scenario_{scenario}/{period["start"]}-{period["end"]}/profiles/"
time_range = parameters.SimulationTimeRange(
    start=datetime.strptime(period["start"], '%Y-%m-%dT%H:%M:%SZ'), 
    end=datetime.strptime(period["end"], '%Y-%m-%dT%H:%M:%SZ'), 
    step=timedelta(seconds=period["step"])
)

orchestrator = Orchestrator(
    datacenters_path=path_in_dc,
    requests_path=None,
    simulation_time_range=time_range,
    scheduling_function=None,
    factor_weights=None
)

y_data_actual, y_data_forecast = load_traces(orchestrator, scenario, period, workload, schedulers=schedulers, seeds=seeds, lc_weights=weights)


In [ ]:
color_blind_palette = [
    "#882255", #0 
    "#AA4499", #1 
    "#CC6677", #2 mix
    "#DDCC77", #3 land
    "#88CCEE", #4 water
    "#44AA99", #5 carbon
    "#117733", #6 
    "#332288"] #7  

In [ ]:
_styles = {
    "actual": "-", 
    "forecast": "--"
}

_colors = {
    "G": colors.IBM_color_blind_palette_RGB[4], 
    "R": colors.IBM_color_blind_palette_RGB[3],
    "RP": colors.IBM_color_blind_palette_RGB[2]
}
_facecolors = {
    "": "None",
    "[1.0, 0.0, 0.0]": color_blind_palette[5],
    "[0.0, 1.0, 0.0]": color_blind_palette[4],
    "[0.0, 0.0, 1.0]": color_blind_palette[3],
    "[0.333, 0.333, 0.334]": color_blind_palette[2]
    

}

_colors = {
    "G": colors.IBM_color_blind_palette_RGB[0], 
    "R": colors.IBM_color_blind_palette_RGB[1],
    "RP": colors.IBM_color_blind_palette_RGB[2]
}

_markers = {
    "": "o",
    "[1.0, 0.0, 0.0]": "s",
    "[0.0, 1.0, 0.0]": "D",
    "[0.0, 0.0, 1.0]": "v",
    "[0.333, 0.333, 0.334]": "^"
    
}

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
def plot(algorithms: list,
         y_data: dict,
         #y_data_mean: dict,
         #y_data_std: dict,
         x_ticks,
         metric: str,
         plot_dir: str,
         log=False,
         #marker_size=40,
         ylim=None,
         boxplot=False):


    fig, ax1 = plt.subplots(nrows=1, ncols=1, figsize=(40, 15)) # figsize=(9, 6.5)
    #ax1.xaxis.set_major_locator(FixedLocator(x_ticks))
    # Step 1: Convert x_ticks (datetime) to numeric
    import matplotlib.dates as mdates
    x_ticks_num = mdates.date2num(x_ticks)

    if boxplot:
        # Draw alternating shaded areas for x-ticks
        #tick_area_width = x_ticks[1] - x_ticks[0]
        tick_area_width = x_ticks_num[1] - x_ticks_num[0]
        boxplot_width = tick_area_width*0.8/len(algorithms)
        for i in range(len(x_ticks_num)):
            if i % 2 == 1:
                ax1.axvspan(x_ticks_num[i] - 0.5*tick_area_width, x_ticks_num[i] + 0.5*tick_area_width, color="#eaeaf2", alpha=0.5, zorder=0)
        sw = [] 
        for i, algorithm in enumerate(algorithms):
            # Distribute positions evenly within each tick area for each algorithm
            positions = [
                x_ticks_num[j] - 0.5 * tick_area_width + (tick_area_width * (i + 1) / (len(algorithms) + 1))
                for j in range(len(x_ticks))
            ]
           
            scheduler, weights = algorithm.split("_") if "_" in algorithm else (algorithm, "")
            sw.append((scheduler, weights))
            bp = ax1.boxplot(
                y_data[algorithm],
                positions=positions,
                widths=boxplot_width,
                patch_artist=True,
                boxprops=dict(
                    facecolor=_facecolors[weights],
                    color=_colors[scheduler],

                ),
                medianprops=dict(color='black'),
                whiskerprops=dict(color=_colors[scheduler]),
                capprops=dict(color=_colors[scheduler]),
                flierprops=dict(marker='o', color=_colors[scheduler], alpha=0.5),
                showmeans=True,
                meanprops=dict(marker='D', markeredgecolor=_colors[scheduler], markerfacecolor=_facecolors[weights]),
                manage_ticks=False
            )

        legend_elements = [
            Patch(facecolor=_facecolors[weights], edgecolor=_colors[scheduler], label=f"{scheduler}_{weights}")
            for scheduler, weights in sw
        ]
        #legend_elements.append(Line2D([0], [0], color='black', lw=2, label='Median'))
        #legend_elements.append(Line2D([0], [0], marker='D', color='w', markerfacecolor='black', markersize=marker_size, label='Mean'))
    
    """else:
        for i in range(len(algorithms)):
            print(f"Algorithm: {algorithms[i]}")
            #print(f"y_data: {y_data[algorithms[i]]}")
            ax1.errorbar(
                x=x_ticks,
                y=y_data_mean[algorithms[i]],
                yerr=y_data_std[algorithms[i]],
                label=PLOT_DICT[algorithms[i]]["label"] + " " + appendix,
                color=PLOT_DICT[algorithms[i]]["color"],
                marker=PLOT_DICT[algorithms[i]]["markers"],
                linestyle=PLOT_DICT[algorithms[i]]["linestyle"],
                linewidth=1.0,
                markersize=marker_size,
                capsize=5,
                elinewidth=1,
                mfc=PLOT_DICT[algorithms[i]]["mfc"])
        # Add custom legend elements for errorbar plots
        legend_elements = [
            Line2D([0], [0],
                   color=PLOT_DICT[algorithms[i]]["color"],
                   marker=PLOT_DICT[algorithms[i]]["markers"],
                   linestyle=PLOT_DICT[algorithms[i]]["linestyle"],
                   markersize=marker_size,
                   label=PLOT_DICT[algorithms[i]]["label"] + " " + appendix)
            for i in range(len(algorithms))
        ]"""

    #ax1.set_xlabel(xlabel=x_label, fontsize=LABEL_SIZE)
    #ax1.set_ylabel(ylabel=metric, fontsize=LABEL_SIZE)
    if log: plt.yscale('log')
    #ax1.tick_params(axis='both', which='major', labelsize=LABEL_SIZE)
    #ax1.set_ylim(ymin=ylim)
    #ax1.yaxis.set_major_formatter(major_formatter)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.legend(
        handles=legend_elements,
        fancybox=True,
        framealpha=0.5,
        handletextpad=0.1,
        columnspacing=0.7,
        #prop={'size': LEGEND_SIZE}, 
        ncol=len(algorithms),# + 2 if boxplot else len(algorithms),
        bbox_to_anchor=(0., 1.02, 1., .102), loc='lower left', mode="expand", borderaxespad=0.
        #bbox_transform=fig.transFigure
        )
    plt.grid(linewidth=0.3)
    plt.tight_layout()
    plot_path = os.path.join(plot_dir, f"{metric.lower()}_footprint.png")
    
    plt.savefig(plot_path)
    plt.clf()
    plt.close()


In [ ]:
def plot_actual_vs_forecast(
        algorithms: list,
        y_data_actual,
        y_data_forecast,
        x_ticks,
        metric: str,
        plot_dir: str
    ):
    fig, ax1 = plt.subplots(nrows=1, ncols=1, figsize=(40, 15)) # figsize=(9, 6.5)
    import matplotlib.dates as mdates
    x_ticks_num = mdates.date2num(x_ticks)

    errors = {}
    for algorithm in algorithms:
        actual = np.array(y_data_actual[algorithm])
        forecast = np.array(y_data_forecast[algorithm])
        errors[algorithm] = forecast - actual
    for i, algorithm in enumerate(algorithms):
        scheduler, weights = algorithm.split("_") if "_" in algorithm else (algorithm, "")
        c = _colors[scheduler]
        m = _markers[weights]
        s = _styles["forecast"]
        data = errors[algorithm]
        mean = np.mean(data, axis=1)
        std = np.std(data, axis=1)
        ax1.errorbar(
            x_ticks,
            mean,
            yerr=std,
            label=f"{algorithm} error",
            color=c,
            marker=m,
            linestyle=s,
            capsize=3
        )

    ax1.set_xlabel("Time")
    ax1.set_ylabel(f"Absolute Error in {metric} Footprint")
    ax1.set_title(f"Forecast vs Actual Absolute Error for {metric}")
    ax1.legend()
    ax1.grid()
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    if not os.path.exists(plot_dir):
        os.makedirs(plot_dir)
    plot_path = os.path.join(plot_dir, f"{metric.lower()}_error.png")
    plt.savefig(plot_path)
    plt.close()


In [ ]:
def plot(footprints, factor, time_range, plot_dir, data_type="actual"):
    import matplotlib.pyplot as plt

    #footprints = {key: value for key, value in sorted(footprints.items())}
    
    # Plot the data
    plt.figure(figsize=(10, 6))
    #colors = plt.cm.tab10.colors  # Use matplotlib's tab10 colormap for distinct colors
    s = _styles[data_type]

    for i, (method, seeds_dict) in enumerate(footprints[data_type].items()):
        # Collect all seeds' footprints for this method
        seeds = sorted(seeds_dict.keys())
        data = np.array([seeds_dict[seed] for seed in seeds])
        mean = np.mean(data, axis=0)
        std = np.std(data, axis=0)
        scheduler, weights = method.split("_") if "_" in method else (method, "")
        c = _colors[scheduler]
        m = _markers[weights]

        plt.errorbar(
            time_range.get_timestamps(),
            mean,
            yerr=std,
            label=f"{method}",
            color=c,
            marker=m,
            linestyle=s,
            capsize=3
        )
    plt.xlabel("Time")
    plt.ylabel(f"Aggregated {factor} Footprint")
    plt.title(f"Aggregated {factor} Footprint Over Time")
    plt.legend()
    plt.grid()
    plt.xticks(rotation=45)
    plt.tight_layout()
    # Save the plot
    if not os.path.exists(plot_dir):
        os.makedirs(plot_dir)
    plot_path = os.path.join(plot_dir, f"{factor.lower()}_footprint.png")
    plt.savefig(plot_path)
    plt.close()


In [ ]:

timestamps = time_range.get_timestamps()
plots = {
    "R": ["G", "R_[1.0, 0.0, 0.0]", "R_[0.0, 1.0, 0.0]", "R_[0.0, 0.0, 1.0]", "R_[0.333, 0.333, 0.334]"],
    "RP": ["G", "RP_[1.0, 0.0, 0.0]", "RP_[0.0, 1.0, 0.0]", "RP_[0.0, 0.0, 1.0]", "RP_[0.333, 0.333, 0.334]"],
    "[1.0, 0.0, 0.0]": ["G", "R_[1.0, 0.0, 0.0]", "RP_[1.0, 0.0, 0.0]"],
    "[0.0, 1.0, 0.0]": ["G", "R_[0.0, 1.0, 0.0]", "RP_[0.0, 1.0, 0.0]"],
    "[0.0, 0.0, 1.0]": ["G", "R_[0.0, 0.0, 1.0]", "RP_[0.0, 0.0, 1.0]"],
    "[0.333, 0.333, 0.334]": ["G", "R_[0.333, 0.333, 0.334]", "RP_[0.333, 0.333, 0.334]"],
}



for plot_name, plot_algorithms in plots.items():
    path_plot = f"experiments/out/scenario_{scenario}/{period['start']}-{period['end']}/workload/{workload}/{period["step"]}/plots/{plot_name}"
    if not os.path.exists(path_plot):
        os.makedirs(path_plot)

    plot(
        algorithms=plot_algorithms,
        y_data=y_data_actual["carbon"],
        x_ticks=timestamps,
        metric="Carbon",
        plot_dir=path_plot,
        boxplot=True
        )
    plot(
        algorithms=plot_algorithms,
        y_data=y_data_actual["water"],
        x_ticks=timestamps,
        metric="Water",
        plot_dir=path_plot,
        boxplot=True
        )
    plot(
        algorithms=plot_algorithms,
        y_data=y_data_actual["land_use"],
        x_ticks=timestamps,
        metric="Land use",
        plot_dir=path_plot,
        boxplot=True
        )
    
    plot_actual_vs_forecast(
        algorithms=plot_algorithms,
        y_data_actual=y_data_actual["carbon"],
        y_data_forecast=y_data_forecast["carbon"],
        x_ticks=timestamps,
        metric="Carbon",
        plot_dir=path_plot,
        boxplot=True
    )
    plot_actual_vs_forecast(
        algorithms=plot_algorithms,
        y_data_actual=y_data_actual["water"],
        y_data_forecast=y_data_forecast["water"],
        x_ticks=timestamps,
        metric="Water",
        plot_dir=path_plot,
        boxplot=True
    )
    plot_actual_vs_forecast(
        algorithms=plot_algorithms,
        y_data_actual=y_data_actual["land_use"],
        y_data_forecast=y_data_forecast["land_use"],
        x_ticks=timestamps,
        metric="Land use",
        plot_dir=path_plot,
        boxplot=True
    )

In [ ]:
from src.parameters import SimulationTimeRange
from datetime import datetime, timedelta

In [ ]:
t = SimulationTimeRange(
    start=datetime.strptime("2024-01-15T00:00:00Z", '%Y-%m-%dT%H:%M:%SZ'), 
    end=datetime.strptime("2024-01-22T00:00:00Z", '%Y-%m-%dT%H:%M:%SZ'), 
    step=timedelta(seconds=60)
)


In [ ]:
path_in_dc = f"./experiments/in/profiles/aws/static/" # provider


In [ ]:
provider = path_in_dc.split("/")[-3] 

In [ ]:
provider

In [ ]:
from src.models.requests import Request
from src.parameters import SimulationTimeRange
from src.models.orchestrator import Orchestrator
import src.models.algorithms as algorithms
from datetime import datetime, timedelta

In [ ]:

provider = "aws"
mae = 0.05
seed = 0


start_str = "2024-07-15T00:00:00Z" 
end_str = "2024-07-22T00:00:00Z"

sim_times = SimulationTimeRange(
    start=datetime.strptime(start_str, '%Y-%m-%dT%H:%M:%SZ'), 
    end=datetime.strptime(end_str, '%Y-%m-%dT%H:%M:%SZ'), 
    step=timedelta(seconds=60)
)

scheduler = "L"
delay_tolerance=0

path_in_dc = f"./experiments/in/profiles/{provider}/static/" # provider
path_in_grids = (f"./experiments/in/profiles/{provider}/dynamic/{mae}/{start_str}-{end_str}/"+"{}.json").format # provider, grid, mae, start, end
schedulers = {
    "L": algorithms.geo_based,
    "R": algorithms.regional_shifting,
    "RP": algorithms.regional_shifting_periodic_jobs,
    "T": algorithms.temporal_shifting,
    "TP": algorithms.temporal_shifting_periodic_jobs,
    "TR": algorithms.regional_and_temporal_shifting,
    "TRP": algorithms.regional_and_temporal_shifting_periodic_jobs
}

lcw = [0.0, 0.0, 1.0] # carbon, water, land use
lwc = {
    "carbon": lcw[0],
    "water": lcw[1],
    "land_use": lcw[2]
}


In [ ]:
orchestrator = Orchestrator(
    datacenters_path=path_in_dc,
    homogeneous=False,
    grid_path=path_in_grids,
    requests_path=None,
    simulation_time_range=sim_times,
    scheduling_function=schedulers[scheduler],
    factor_weights=lwc, 
    delay_tolerance=timedelta(minutes=delay_tolerance)
)

In [ ]:
arrival_time = datetime.strptime("2024-07-21T00:33:00Z", '%Y-%m-%dT%H:%M:%SZ')

req = Request(
    simulation_time_range=sim_times,
    id=4317,
    arrival_location="eu-north-1",
    VM_instance="m4.xlarge",
    n_nodes=8,
    input_size_bytes=6104074583,
    arrival_time=arrival_time,
    runtime_sec=timedelta(seconds=1600.157),
    avg_cpu_usr_util=71.74010076775431,
    deadline=arrival_time + timedelta(minutes=60*4)
)
#4317,6104074583,m4.xlarge,8,1600.157,71.74010076775431,eu-north-1,33,

In [ ]:
orchestrator.datacenters

In [ ]:
from src.models.objectives import evaluate_footprint

evaluate_footprint(
    t_0 = req.arrival_time, 
    r = req, 
    d = orchestrator.datacenters[req.arrival_location], 
    o = orchestrator
    )

In [ ]:
evaluate_footprint(
    t_0 = req.arrival_time, 
    r = req, 
    d = orchestrator.datacenters["eu-west-1"], 
    o = orchestrator
    )

In [ ]:
req.execution_and_tracing(
    d=orchestrator.datacenters["eu-west-1"],
    t_0=req.arrival_time,
    o=orchestrator
)

In [ ]:
req.trace.get_csv_lines(o=orchestrator)